##Load Silver data

Reads the cleaned, deduplicated vessel position data from the Silver Delta
table as the foundation for Gold layer business metrics.

In [0]:
silver_table = "logistics_pipeline.silver.vessel_positions_clean"

df_silver = spark.table(silver_table)

display(df_silver)
print(f"Silver rows loaded: {df_silver.count()}")

## Aggregating into 5 minute snapshot windows

Individual AIS broadcasts are seconds apart which is too granular for a congestion
signal. Rounds broadcast_timestamp into 5 minute buckets (matching our
ingestion poll interval) so each bucket represents one meaningful snapshot
of port activity, then computes summary stats per snapshot:
- total vessel count
- count of "stationary but not moored" vessels (sog near 0, nav_status != Moored), a proxy for vessels waiting/blocked
- average speed over ground across all vessels

**Definition refined empirically:** an initial version flagged ~20% of
vessels as "stationary and not moored" every snapshot - implausibly high
and consistent. Investigating the actual nav_status_label breakdown showed
"Under way using engine" (a genuine status/speed contradiction) was the
dominant driver, but "At anchor" and "Not reported" were also being
counted, despite representing legitimate or unknown states rather than
congestion. The definition now explicitly excludes Moored, At anchor, and
Not reported.

**Known limitation:** `nav_status` is null for ~57% of vessel records
(verified in the Silver notebook's null audit). Since this metric depends
on nav_status_label, the congestion signal is effectively built from the
~43% of vessels that report a status and not the full tracked fleet. This is
a real constraint of the data source, not a bug.

In [0]:
from pyspark.sql.functions import window, count, avg, when, sum as spark_sum, col


# Statuses where being stationary is NORMAL, not a congestion signal -
# excluded from our "waiting/blocked" proxy. Confirmed via real data check:
# "At anchor" and "Not reported" were not the dominant driver, but both
# represent legitimate/unknown states that shouldn't count as congestion.
NORMAL_STATIONARY_STATUSES = ["Moored", "At anchor", "Not reported"]

df_snapshots = (
    df_silver
    # Flag vessels barely moving (sog < 0.5 knots) whose nav_status_label
    # is NOT one of the normal to be stationary states. e.g. a vessel
    # marked "Under way using engine" that isn't actually moving is a
    # genuine contradiction and one of the strongest congestion signal
    .withColumn(
        "is_stationary_not_moored",
        when(
            (col("sog") < 0.5) & (~col("nav_status_label").isin(NORMAL_STATIONARY_STATUSES)),
            1
        ).otherwise(0)
    )
    .groupBy(window(col("broadcast_timestamp"), "5 minutes").alias("snapshot_window"))
    .agg(
        count("*").alias("vessel_count"),
        spark_sum("is_stationary_not_moored").alias("stationary_not_moored_count"),
        avg("sog").alias("avg_speed_over_ground")
    )
    .withColumn(
        "stationary_not_moored_pct",
        (col("stationary_not_moored_count") / col("vessel_count") * 100)
    )
    .orderBy("snapshot_window")
)

display(df_snapshots)

In [0]:
display(
    df_silver
    .filter((col("sog") < 0.5) & (col("nav_status_label") != "Moored"))
    .groupBy("nav_status_label")
    .count()
    .orderBy(col("count").desc())
)

## Nav status composition per snapshot

Complements the congestion signal with a compositional view, how many
vessels are moored, underway, at anchor, etc. at each snapshot. Useful for
a dashboard showing overall port activity, not just the risk indicator.

In [0]:
from pyspark.sql.functions import window as spark_window

df_status_breakdown = (
    df_silver
    # Same 5 minute windowing as the congestion snapshot, so both Gold
    # tables can be joined/compared on snapshot window later if needed
    .groupBy(
        spark_window(col("broadcast_timestamp"), "5 minutes").alias("snapshot_window"),
        col("nav_status_label")
    )
    .count()
    .withColumnRenamed("count", "vessel_count")
    .orderBy("snapshot_window", col("vessel_count").desc())
)

display(df_status_breakdown)

## Writing Gold tables

Persists both business facing metrics as Delta tables. The congestion
snapshot signal and the nav status composition breakdown. 

In [0]:
congestion_table = "logistics_pipeline.gold.congestion_snapshots"
status_table = "logistics_pipeline.gold.nav_status_breakdown"

# overwrite mode for now, same reasoning as Silver - Bronze ingestion isn't
# automated yet, so there's no genuinely incremental data to append safely
df_snapshots.write.mode("overwrite").saveAsTable(congestion_table)
df_status_breakdown.write.mode("overwrite").saveAsTable(status_table)

print(f"Written to {congestion_table}: {df_snapshots.count()} rows")
print(f"Written to {status_table}: {df_status_breakdown.count()} rows")